<a href="https://colab.research.google.com/github/nivethithanm/python-systems-full/blob/main/DEEP_01_memory_model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# DEEP-01: Python's Memory Model — What Actually Happens

**Time**: ~3 hours | **Depth**: First principles

**Systems mapping**: Python variables → pointers → heap objects → reference counting → GC cycles

You'll build: a reference counting garbage collector from scratch.

In [1]:
import sys
import gc
import ctypes
import weakref

## 1. Variables are pointers, not boxes

Python → C mapping: `x = 42` does NOT store 42 in a box called x. It creates an int object on the heap and makes x point to it.

In [2]:
# Exercise 1.1: Prove that variables are pointers
a = [1, 2, 3]
b = a  # b is NOT a copy. It's a second pointer to the SAME list.

# YOUR PREDICTION before running: what does b[0] become?
b[0] = 999

# Verify
assert a[0] == 999  # fill in your prediction
assert id(a) == id(b)  # same object, same memory address
print(f'a is b: {a is b}')  # 'is' checks identity (same pointer), not equality

a is b: True


In [3]:
# Exercise 1.2: Trace the reference count
# sys.getrefcount() returns refcount (adds 1 for the argument itself)

x = 'hello'
print(f'refcount after creation: {sys.getrefcount(x)}')  # likely 2+ (interned strings)

y = x  # another reference
print(f'refcount after y=x: {sys.getrefcount(x)}')

z = [x, x, x]  # 3 more references
print(f'refcount after z=[x,x,x]: {sys.getrefcount(x)}')

del y
print(f'refcount after del y: {sys.getrefcount(x)}')

# YOUR TASK: predict each refcount before running, then verify
# Write your predictions as comments above each print

refcount after creation: 4294967295
refcount after y=x: 4294967295
refcount after z=[x,x,x]: 4294967295
refcount after del y: 4294967295


In [4]:
# Exercise 1.3: Mutable vs immutable — the REAL difference
# Immutable: int, str, tuple → operations create NEW objects
# Mutable: list, dict, set → operations modify IN PLACE

# YOUR TASK: For each operation below, predict:
#   (a) Does id(x) change? (new object or same?)
#   (b) What is the final value?

# Case 1:
x = 10
old_id = id(x)
x += 1
assert (id(x) == old_id) == False  # True or False?

# Case 2:
x = [1, 2]
old_id = id(x)
x += [3]  # += on list does extend IN PLACE
assert (id(x) == old_id) == True  # True or False?

# Case 3:
x = (1, 2)
old_id = id(x)
x += (3,)  # += on tuple creates NEW tuple
assert (id(x) == old_id) == False  # True or False?

print('✅ 1.3 done — you understand the mutable/immutable pointer model')

✅ 1.3 done — you understand the mutable/immutable pointer model


## 2. Build a Reference Counting GC from Scratch

CPython uses reference counting as its primary GC. When refcount hits 0, the object is freed immediately. You'll simulate this.

In [9]:
# Exercise 2.1: Implement a mini heap with reference counting

class MiniHeap:
    """Simulates Python's memory model.

    Each 'object' is an entry in self.objects: {obj_id: {value, refcount}}
    Variables are entries in self.variables: {name: obj_id}
    """

    def __init__(self):
        self.objects = {}  # obj_id → {value, refcount}
        self.variables = {}  # var_name → obj_id
        self.next_id = 0
        self.freed = []  # track freed object IDs

    def allocate(self, value) -> int:
        """Create a new object on the heap. Return its ID."""
        # YOUR CODE HERE
        # Assign self.next_id, store {value, refcount: 0}, increment next_id
        curr_id = self.next_id
        self.objects[curr_id] = {
            'value': value,
            'refcount': 0,
            'references': []
        }
        self.next_id += 1
        return curr_id

    def assign(self, var_name: str, obj_id: int):
        """Assign a variable to point at an object.

        If var_name already pointed somewhere, DECREF the old target.
        INCREF the new target.
        """
        if var_name in self.variables:
            self._decref(self.variables[var_name])
        self.variables[var_name] = obj_id
        self._incref(obj_id)

    def delete_var(self, var_name: str):
        """Delete a variable (like 'del x').
        DECREF the target. Remove from variables.
        """
        # YOUR CODE HERE
        self._decref(self.variables.pop(var_name))

    def _decref(self, obj_id: int):
        """Decrement refcount. If 0, FREE the object."""
        # YOUR CODE HERE
        # If refcount reaches 0: remove from self.objects, add to self.freed
        if self.objects[obj_id]['refcount'] > 0:
            self.objects[obj_id]['refcount'] -= 1
        if self.objects[obj_id]['refcount'] == 0:
            self.objects.pop(obj_id)
            self.freed.append(obj_id)

    def _incref(self, obj_id: int):
        """Increment refcount."""
        # YOUR CODE HERE
        self.objects[obj_id]['refcount'] += 1

    def get_value(self, var_name: str):
        """Get the value a variable points to."""
        # YOUR CODE HERE
        obj_id = self.variables[var_name]
        return self.objects[obj_id]['value']

    def stats(self) -> dict:
        return {
            'live_objects': len(self.objects),
            'variables': len(self.variables),
            'freed': len(self.freed)
        }

# Test
heap = MiniHeap()

# x = 'hello'
obj1 = heap.allocate('hello')
heap.assign('x', obj1)
assert heap.get_value('x') == 'hello'
assert heap.objects[obj1]['refcount'] == 1

# y = x  (second pointer to same object)
heap.assign('y', obj1)
assert heap.objects[obj1]['refcount'] == 2

# del x
heap.delete_var('x')
assert heap.objects[obj1]['refcount'] == 1  # y still points to it

# del y → refcount 0 → freed!
heap.delete_var('y')
assert obj1 not in heap.objects  # freed!
assert obj1 in heap.freed

print(f'Stats: {heap.stats()}')
print('✅ 2.1 — you built reference counting GC')

Stats: {'live_objects': 0, 'variables': 0, 'freed': 1}
✅ 2.1 — you built reference counting GC


In [12]:
# Exercise 2.2: The cycle problem
# Reference counting FAILS with cycles: A → B → A, both refcount=1 forever.

# Demonstrate the problem with your MiniHeap:
# 1. Create two objects
# 2. Make them reference each other (add a 'references' field to objects)
# 3. Delete both variables
# 4. Show that neither gets freed (refcount never hits 0)

# Then implement a simple mark-and-sweep cycle collector:
def collect_cycles(heap: MiniHeap) -> int:
    """Find and free objects involved in reference cycles.

    Simple mark-and-sweep:
    1. MARK: Start from all variable roots, mark reachable objects
    2. SWEEP: Any unmarked object with refcount > 0 is in a cycle → free it

    Return number of freed objects.
    """
    # YOUR CODE HERE
    reachable = set()

    def walk(obj_id):
        if obj_id in reachable:
            return
        reachable.add(obj_id)
        # Follow pointers to other objects
        for ref_id in heap.objects[obj_id].get('references', []):
            walk(ref_id)

    # Start the walk from every variable currently in use (roots)
    for obj_id in heap.variables.values():
        walk(obj_id)

    # 2. SWEEP: Anything in heap.objects NOT in reachable is a cycle/trash
    to_free = [oid for oid in heap.objects if oid not in reachable]

    num_freed = len(to_free)
    for oid in to_free:
        heap.objects.pop(oid)
        heap.freed.append(oid)

    return num_freed


heap = MiniHeap()

# 1. Create two objects
id_a = heap.allocate("Object A")
id_b = heap.allocate("Object B")

# 2. Point variables at them
heap.assign('alice', id_a)
heap.assign('bob', id_b)

# 3. Create the CYCLE: A points to B, B points to A
heap.objects[id_a]['references'].append(id_b)
heap.objects[id_b]['references'].append(id_a)
# Manually simulate the internal refcount increase for these links
heap._incref(id_a)
heap._incref(id_b)

# 4. Delete the variable "roots"
heap.delete_var('alice')
heap.delete_var('bob')

# BUG: Refcounts are still 1 because they point to each other!
print(f"Refcount A: {heap.objects[id_a]['refcount']}") # Still 1
print(f"Refcount B: {heap.objects[id_b]['refcount']}") # Still 1

# 5. Run your new collector to save the day
freed_count = collect_cycles(heap)
print(f"Cycles cleaned: {freed_count}") # Should be 2

Refcount A: 1
Refcount B: 1
Cycles cleaned: 2


## 3. Shallow vs Deep Copy — Implement Both

In [13]:
# Exercise 3.1: Implement shallow_copy and deep_copy for nested structures

def shallow_copy(obj):
    """Create a new container with the SAME element references.
    Support: list, dict, set. For other types, return as-is.
    Do NOT use copy module.
    """
    # YOUR CODE HERE
    if isinstance(obj, list):
        return obj[:] # Slice creates a new list with same items
    if isinstance(obj, dict):
        return obj.copy() # Built-in dict copy is shallow
    if isinstance(obj, set):
        return obj.copy() # Built-in set copy is shallow
    return obj # Immutable types (int, str, etc.) don't need copying

def deep_copy(obj, memo=None):
    """Create a new container with recursively copied elements.
    Handle: list, dict, set, tuple.
    Handle circular references using memo dict (obj_id → copy).
    Do NOT use copy module.
    """
    # YOUR CODE HERE
    # Hint: check id(obj) in memo to handle cycles
    if memo is None:
        memo = {}

    # 1. If we've seen this object before, return its existing copy
    obj_id = id(obj)
    if obj_id in memo:
        return memo[obj_id]

    # 2. Handle Lists
    if isinstance(obj, list):
        new_list = []
        memo[obj_id] = new_list # Register BEFORE recursing to handle cycles
        for item in obj:
            new_list.append(deep_copy(item, memo))
        return new_list

    # 3. Handle Dicts
    if isinstance(obj, dict):
        new_dict = {}
        memo[obj_id] = new_dict
        for k, v in obj.items():
            new_dict[deep_copy(k, memo)] = deep_copy(v, memo)
        return new_dict

    # 4. Handle Tuples (Tuples are immutable, but can contain mutable lists!)
    if isinstance(obj, tuple):
        # We can't "pre-register" a tuple because it's immutable,
        # but tuples can't be part of a cycle themselves unless they contain a list.
        new_tuple = tuple(deep_copy(item, memo) for item in obj)
        memo[obj_id] = new_tuple
        return new_tuple

    # 5. Handle Sets
    if isinstance(obj, set):
        new_set = set()
        memo[obj_id] = new_set
        for item in obj:
            new_set.add(deep_copy(item, memo))
        return new_set

    # Base case: immutable types (int, str, None) just return themselves
    return obj


# Test shallow
orig = [[1, 2], [3, 4]]
shallow = shallow_copy(orig)
shallow[0][0] = 999
assert orig[0][0] == 999  # shared inner list!
assert id(shallow) != id(orig)  # different outer list

# Test deep
orig2 = [[1, 2], [3, 4]]
deep = deep_copy(orig2)
deep[0][0] = 999
assert orig2[0][0] == 1  # independent!

# Test cycle handling
cyclic = [1, 2]
cyclic.append(cyclic)  # self-referencing!
deep_cyclic = deep_copy(cyclic)
assert deep_cyclic[2] is deep_cyclic  # cycle preserved in copy
assert deep_cyclic[2] is not cyclic  # but it's a NEW cycle

print('✅ 3.1 — shallow vs deep copy from scratch')

✅ 3.1 — shallow vs deep copy from scratch


## Mastery Check

If you truly understand this, you should be able to:
- [ ] Explain why `a = b` never copies data in Python
- [ ] Predict refcount changes for any sequence of assignments
- [ ] Explain why `+=` behaves differently on lists vs tuples
- [ ] Implement a working cycle collector
- [ ] Explain why Python needs both refcounting AND mark-and-sweep